In [0]:
%sql
-- capability_latency_baseline
-- Two guards, both learned the hard way:
--   n_samples >= 30      — dsa_batch_summary had n=1, so p50=p95=p99=bound and 2826ms fired
--                          as an "anomaly" when dsa_copilot's p95 is 3963ms
--   baseline_span_days >= 7 — the Jun 12 → Aug 12 blackout means a nominal 30-day window holds
--                          only ~1.5 days of data, so n alone is not evidence of stability
CREATE OR REPLACE TABLE mq_gmdf_dev.oil_obs.capability_latency_baseline AS
SELECT
    b.capability,
    approx_percentile(b.latency_ms, 0.50) AS p50_ms,
    approx_percentile(b.latency_ms, 0.95) AS p95_ms,
    approx_percentile(b.latency_ms, 0.99) AS p99_ms,
    approx_percentile(b.latency_ms, 0.95)
      + 3 * (approx_percentile(b.latency_ms, 0.75)
             - approx_percentile(b.latency_ms, 0.25))   AS anomaly_upper_bound_ms,
    count(*)                                            AS n_samples,
    datediff(max(b.called_at), min(b.called_at))        AS baseline_span_days,
    (count(*) >= 30
     AND datediff(max(b.called_at), min(b.called_at)) >= 7) AS is_reliable,
    min(b.called_at)                                    AS baseline_from,
    max(b.called_at)                                    AS baseline_through,
    current_timestamp()                                 AS computed_at
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.called_at >= current_timestamp() - INTERVAL 30 DAYS
  AND b.is_credential_fastfail = false
  AND b.transport             <> 'mock'
  AND b.success                = true
GROUP BY b.capability;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- response_schema_baseline
-- The old upper bound (current_timestamp() - INTERVAL 1 DAYS) excluded every dsa_* capability,
-- since that data begins 2026-08-18 01:30. That is why the baseline covered 3 of 5.
CREATE TABLE IF NOT EXISTS mq_gmdf_dev.oil_obs.response_schema_baseline (
    capability       STRING,
    schema_ddl       STRING,
    sample_count     BIGINT,
    baseline_from    TIMESTAMP,
    baseline_through TIMESTAMP
);

INSERT OVERWRITE TABLE mq_gmdf_dev.oil_obs.response_schema_baseline
SELECT
    b.capability,
    schema_of_json_agg(cast(b.response_parsed AS STRING)) AS schema_ddl,
    count(*)         AS sample_count,
    min(b.called_at) AS baseline_from,
    max(b.called_at) AS baseline_through
FROM mq_gmdf_dev.oil_obs.v_llm_bronze b
JOIN mq_gmdf_dev.oil_obs.capability_registry r
  ON r.capability = b.capability AND r.active = true
WHERE b.success = true
  AND b.is_blank_output        = false
  AND b.is_credential_fastfail = false
  AND b.called_at >= current_timestamp() - INTERVAL 30 DAYS
GROUP BY b.capability
HAVING count(*) >= 20;

num_affected_rows,num_inserted_rows
1,1
